In [ ]:
# Set up SAM2 for segmentation

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.sam2_image_predictor import SAM2ImagePredictor # initially used SAM2AutomaticMaskGenerator, but was too slow

device = torch.device("mps")

CHECKPOINT = "checkpoints/sam2_hiera_large.pt"
MODEL_CFG = "configs/sam2/sam2_hiera_l.yaml"

sam2_model = build_sam2(MODEL_CFG, CHECKPOINT, device=device)

predictor = SAM2ImagePredictor(sam2_model)

Segmentation: separating the body from the background via SAM2

In [ ]:
def segment(image_path, output_path=None):
    """
    Take in an image, run SAM2, and return best person mask
    """
    # Load image
    image = Image.open(image_path).convert("RGB")
    image_array = np.array(image)

    # Get masks
    l, w = image_array.shape[:2] # length, width
    input_pts = np.array([
        [w // 2, l // 2], # center of photo 
        [w // 2, 2 * l // 5], # upper body 
        [w // 2, 4 * l // 5], # lower body 
        [w // 3, l // 2], # left side of body 
        [2 * w // 3, l // 2], # right side of body 
        [2, 2], # top-left corner, background 
        [w - 2, l - 2], # bottom-right corner, background 
        [2, l - 2], # bottom-left corner, background 
        [w - 2, 2] # top-right corner, background 
    ])
    labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0]) # last four input points are background
    predictor.set_image(image_array)
    masks_raw, scores, low_res_masks_logits = predictor.predict(point_coords=input_pts, point_labels=labels)
    masks = [{'segmentation': masks_raw[i].astype(bool), 'area': masks_raw[i].sum()} for i in range(len(masks_raw))]

    # Sort by mask area
    masks = sorted(masks, key=lambda m: m['area'], reverse=True)

    # Select mask that most likely matches the human figure
    # (i.e. large area but not the largest [would likely be the entire background])
    area = image_array.shape[0] * image_array.shape[1] # height * width
    silhouette_mask = None

    for mask in masks:
        coverage = mask['area'] / area
        if coverage < 0.70 : # max out at 70% coverage bc don't want background; want biggest area otherwise; masks already sorted by area
            silhouette_mask = mask
            break
    if silhouette_mask is None: # take largest area otherwise 
        silhouette_mask = masks[0]

    silhouette = silhouette_mask['segmentation']

    # fill holes
    silhouette_uint8 = silhouette.astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (10, 10)) # make a kernel in ellipse size (humans not boxy) to pass over
    silhouette_uint8 = cv2.morphologyEx(silhouette_uint8, cv2.MORPH_CLOSE, k)

    # get rid of small background noise 
    n_blobs, labels, stats, coords = cv2.connectedComponentsWithStats(silhouette_uint8)
    block_areas = stats[1:, cv2.CC_STAT_AREA] # get array of areas for every block (excl. background)
    biggest_label = 1 + np.argmax(block_areas) # select idx of largest area (silhouette)
    silhouette_uint8 = np.where(labels == biggest_label, 255, 0).astype(np.uint8) # if label belongs to silhouette, set to 255
    silhouette = silhouette_uint8.astype(bool)
    
    # Put mask over image
    masked_img = image_array.copy()
    masked_img[~silhouette] = 0  # black out background

    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_array)
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(silhouette, cmap='gray')
    axes[1].set_title("Raw Mask")
    axes[1].axis('off')

    axes[2].imshow(masked_img)
    axes[2].set_title("Human Silhouette")
    axes[2].axis('off')

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
    plt.show()

    return silhouette, masked_img

In [ ]:
# apply segment() to all photos

from pathlib import Path 

folder_path = Path("/Users/hannahlatif/Documents/CS131/Final_Project/Milestone/og_photos")
for img_path in folder_path.glob("*.jpg"):
    segment(img_path, Path(f"/Users/hannahlatif/Documents/CS131/Final_Project/Milestone/segmented/{img_path.stem}_segmented{img_path.suffix}"))

Pose Keypoint Detection via MediaPipe BlazePose

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

In [ ]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions
import mediapipe as mp

options = PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="checkpoints/pose_landmarker.task"),
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1)

pose_landmarker = PoseLandmarker.create_from_options(options)

In [ ]:
def detect_pose(image_path):
    """
    Detect key body points an image via MediaPipe BlazePose, 
    returning coordinates and an annotated version of the image. 
    """
    # Load image
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # Make image MediaPipe-friendly
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_np)

    # Pose detection
    pose = pose_landmarker.detect(mp_image)

    if not pose.pose_landmarks:
        print(f"No person found in {image_path}")
        return None, image_np

    # Get (x, y) pixel coordinates of each landmark
    h, w = image_np.shape[:2]
    figure = pose.pose_landmarks[0]  
    keypoints = { # convert normalized dimensions to coordinates
        'nose': (int(figure[0].x * w),  int(figure[0].y * h)),
        'left shoulder': (int(figure[11].x * w), int(figure[11].y * h)),
        'right shoulder': (int(figure[12].x * w), int(figure[12].y * h)),
        'left elbow': (int(figure[13].x * w), int(figure[13].y * h)),
        'right elbow': (int(figure[14].x * w), int(figure[14].y * h)),
        'left hip': (int(figure[23].x * w), int(figure[23].y * h)),
        'right hip': (int(figure[24].x * w), int(figure[24].y * h)),
        'left knee': (int(figure[25].x * w), int(figure[25].y * h)),
        'right knee': (int(figure[26].x * w), int(figure[26].y * h))}

    # Annotate image
    annotated = image_np.copy()
    for name, (x, y) in keypoints.items():
        cv2.circle(annotated, (x, y), 9, (0, 0, 255), -1) # filled circle at each keypoint
        cv2.putText(annotated, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image_np); axes[0].set_title("OG"); axes[0].axis('off')
    axes[1].imshow(annotated); axes[1].set_title("With Keypoints"); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    return keypoints, annotated


In [ ]:
for img_path in folder_path.glob("*.jpg"):
    detect_pose(img_path)


Pose Normalization: scale the silhouette so the torso length is consistent across photos (300 px) and the hips sit at the middle of the canvas.

In [ ]:
def normalize_body(silhouette, keypoints):
    """
    Takes in the segmented figure of the orignal photo 
    and the keypoint anatomical markers to normalize the body
    """
    
    h, w = silhouette.shape[:2]

    # Get reference points based on stationary body points (shoulder and hip)
    mid_shoulder = ((keypoints['left shoulder'][0] + keypoints['right shoulder'][0]) // 2, # x coordinate
                    (keypoints['left shoulder'][1] + keypoints['right shoulder'][1]) // 2) # y coordinate
    mid_hip = ((keypoints['left hip'][0] + keypoints['right hip'][0]) // 2, # x coordinate
               (keypoints['left hip'][1] + keypoints['right hip'][1]) // 2) # y coordinate

    # Torso length
    torso_l = np.sqrt((mid_shoulder[0] - mid_hip[0])**2 + (mid_shoulder[1] - mid_hip[1])**2) # take Euclidean distance 

    # Scaling factor (divide by relative torso length and resize to simplify heatmap function)
    scale = 400 / torso_l

    # Scale silhouette and hip
    scaled_h = int(h * scale)
    scaled_w = int(w * scale)
    scaled = cv2.resize(silhouette.astype(np.uint8) * 255, (scaled_w, scaled_h))
    scaled_mid_hip = (int(mid_hip[0] * scale), int(mid_hip[1] * scale))

    # Set fixed black canvas size for comparison
    canvas_h, canvas_w = 1200, 600
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)

    # Offset: place mid-hip at center
    x_off = canvas_w // 2 - scaled_mid_hip[0]
    y_off = canvas_h // 2 - scaled_mid_hip[1]

    # Put scaled silhouette on canvas. First make sure in bounds
    x1 = max(0, x_off)
    y1 = max(0, y_off)
    x2 = min(canvas_w, x_off + scaled_w)
    y2 = min(canvas_h, y_off + scaled_h)

    x3 = max(0, -x_off) # adjust if silhouette extends beyond canvas
    y3 = max(0, -y_off) # adjust if silhouette extends beyond canvas
    x4 = x3 + x2 - x1
    y4 = y3 + y2 - y1
    canvas[y1:y2, x1:x2] = scaled[y3:y4, x3:x4]

    return canvas.astype(bool), scale, (x_off, y_off)


In [ ]:
# Test pipeline 

for img_path in folder_path.glob("*.jpg"):
    silhouette_p, _ = segment(img_path)
    keypoints_p, _ = detect_pose(img_path)
    normalized_p, scale_p, offset_p = normalize_body(silhouette_p, keypoints_p)

    plt.imshow(normalized_p, cmap='gray')
    plt.title("Normalized Silhouette")
    plt.axis('off')
    plt.show()

Proportion Measurement and Calculation

In [ ]:
def og_to_canvas(pt, scale, offset):
    """
    Helper function to convert a point on the original photo 
    to a point on the normalized canvas
    """
    return (
        int(pt[0] * scale) + offset[0],
        int(pt[1] * scale) + offset[1]
    )

In [ ]:
def largest_whitespace(white_cols):
    """
    Identify the largest contiguous white space given an array
    of white pixel column indicies for a row (purpose is to 
    ignore arms in measurement of chest, hips, etc.)
    """ 
    if len(white_cols) == 0:
        return 0
    
    max_width = 0
    current_width = 1
    
    for i in range(1, len(white_cols)):
        if white_cols[i] == white_cols[i-1] + 1:  # contiguous
            current_width += 1
        else:  # if gap, restart
            if current_width > max_width:
                max_width = current_width
            current_width = 1
    
    # After reacing last pixel
    if current_width > max_width:
        max_width = current_width
    
    return max_width

In [ ]:
def measure_proportions(normalized_silhouette, keypoints, scale, offset):
    """
    Measure body proportions at horizontal slices 
    and return proportion ratios of widths relative 
    to torso length and the y-coordinate for keypoints.
    """
    h, w = normalized_silhouette.shape[:2]
    silhouette_uint8 = normalized_silhouette.astype(np.uint8) * 255    

    # Get canvas point for key points
    l_shoulder = og_to_canvas(keypoints['left shoulder'], scale, offset)
    r_shoulder = og_to_canvas(keypoints['right shoulder'], scale, offset)
    l_hip = og_to_canvas(keypoints['left hip'], scale, offset)
    r_hip = og_to_canvas(keypoints['right hip'], scale, offset)
    l_elbow  = og_to_canvas(keypoints['left elbow'], scale, offset)
    r_elbow = og_to_canvas(keypoints['right elbow'], scale, offset)


    # Calculate torso length via distance from shoulders to hips 
    mid_shoulder_y = (l_shoulder[1] + r_shoulder[1]) // 2
    mid_hip_y = (l_hip[1] + r_hip[1]) // 2
    torso_length = mid_hip_y - mid_shoulder_y

    # Get the y coordinate for key points
    y_keypts = {'shoulders': mid_shoulder_y,
                'chest': mid_shoulder_y + int(torso_length * 0.27), # chest sits around a quarter of the way down the length btwn shoulder and hips
                'waist': mid_shoulder_y + int(torso_length * 0.58),  # estimate waist to sit around .58 of torso
                'hips': mid_hip_y}

    # bind waist and chest measurements by elbow boundaries 
    x_left  = min(l_elbow[0], r_elbow[0])
    x_right = max(l_elbow[0], r_elbow[0])
    x_left  = max(0, x_left)
    x_right = min(w, x_right)

    ratios = {}

    # Get ratios and widths of key points
    for keypt, y in y_keypts.items():
        if 0 <= y < h:
            row = silhouette_uint8[y, :] # silhouette_uint8[y, x_left:x_right] if keypt == 'waist' else 
            white_cols = np.where(row > 127)[0] # get indicies of white pixels
            if len(white_cols) > 0:
                width = largest_whitespace(white_cols)
                ratios[keypt] = width / torso_length
            else: # safety net 
                ratios[keypt] = 0

    # See slices on silhouette in different colors 
    vis = cv2.cvtColor(silhouette_uint8, cv2.COLOR_GRAY2BGR)
    colors = {'shoulders': (255, 0, 0), 
              'chest': (0, 255, 0), 
              'waist': (0, 0, 255), 
              'hips': (255, 100, 0)}
    for keypt, y in y_keypts.items():
        if 0 <= y < h:
            cv2.line(vis, (0, y), (w, y), colors[keypt], 2)
            cv2.putText(vis, f"{keypt}: {ratios[keypt]:.4f}", (5, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colors[keypt], 2)

    plt.figure(figsize=(7, 11))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title("Proportion Measurement Slices")
    plt.axis('off')
    plt.show()

    return ratios, y_keypts


In [ ]:
# run on images

for img_path in folder_path.glob("*.jpg"):
    silhouette_, masked_img = segment(img_path)
    keypoints_, annotated = detect_pose(img_path)
    normalized_, scale_, offset_ = normalize_body(silhouette_, keypoints_)
    proportions_, y_keypts_ = measure_proportions(normalized_, keypoints_, scale_, offset_)
    print(proportions_)

Heatmap: show regions of growth and loss

In [ ]:
def heatmap(image_path_a, image_path_b):
    """
    Compare two progress photos and create a heatmap with
    body proportion changes between them
    Green = growth, Red = loss, Blue = no change
    _b = before, _a = after
    """

    # before image
    silhouette_b, masked_b = segment(image_path_b)
    keypoints_b, annotated_b = detect_pose(image_path_b)
    normalized_b, scale_b, offset_b = normalize_body(silhouette_b, keypoints_b)
    ratios_b, y_keypts_b = measure_proportions(normalized_b, keypoints_b, scale_b, offset_b)

    # after image
    silhouette_a, masked_a = segment(image_path_a)
    keypoints_a, annotated_a = detect_pose(image_path_a)
    normalized_a, scale_a, offset_a = normalize_body(silhouette_a, keypoints_a)
    ratios_a, y_keypts_a = measure_proportions(normalized_a, keypoints_a, scale_a, offset_a)

    # Compute proportion differences (after - before)
    diff = {k: ratios_a[k] - ratios_b[k] for k in ratios_a}

    # Create heatmap overlay on before photo's silhouette
    heatmap = cv2.cvtColor(normalized_b.astype(np.uint8) * 255, cv2.COLOR_GRAY2BGR)

    # Define regions for shading
    shoulders_b, chest_b, waist_b, hips_b = y_keypts_b.values()
    torso_len = hips_b - shoulders_b
    y_shoulder_cutoff = max(0, shoulders_b - int(torso_len * 0.2)) # give the shoulders a region to show its proportion changes
    y_hips_cuotff = min(normalized_b.shape[0], hips_b + int(torso_len * 0.15)) # bottom cutoff to show hip changes
    slice_regions = [
        ('shoulders', y_shoulder_cutoff, chest_b),
        ('chest', chest_b, waist_b),
        ('waist', waist_b, hips_b),
        ('hips', hips_b, y_hips_cuotff)
    ]

    # Color regions
    change_threshold = 0.02 # if ratio changes by more or less than this amount, note change. Otherwise, note no change
    for section, y_start, y_end in slice_regions:
        change = diff[section]
        if change > change_threshold:
            color = (0, 200, 0) # green = growth
        elif change < -change_threshold:
            color = (0, 0, 200)  # red = loss
        else:
            color = (0, 0, 200) # blue = no change

        region = heatmap[y_start:y_end, :]
        mask = normalized_b[y_start:y_end, :] > 127 # color only the silhouette 
        region[mask] = color
        heatmap[y_start:y_end, :] = region

    # Draw slice lines and labels
    for name, y_start, y_end in slice_regions:
        change = diff[name]
        cv2.line(heatmap, (0, y_start), (normalized_b.shape[1], y_start), (255, 255, 255), 1)
        cv2.putText(heatmap, f"{name}: {change:+.3f}", (5, y_start + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)

    # Plot side by side
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))
    axes[0].imshow(Image.open(image_path_a)); axes[0].set_title("Photo A (After)"); axes[0].axis('off')
    axes[1].imshow(Image.open(image_path_b)); axes[1].set_title("Photo B (Before)"); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)); axes[2].set_title("Change Heatmap"); axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    # Print proportion table
    print("\nProportion Changes:")
    print(f"{'Region':<12} {'Before (B)':>12} {'After (A)':>12} {'Change':>10} {'Direction':>12}")
    print("-" * 60)
    for name in ratios_a:
        a, b, d = ratios_a[name], ratios_b[name], diff[name]
        direction = "increased" if d > change_threshold else "decreased" if d < change_threshold else "no detectable change"
        print(f"{name:<12} {b:>12.3f} {a:>12.3f} {d:>+10.3f} {direction:>12}")

    return diff, ratios_a, ratios_b


Final Pipeline Execution: segmentation/keypoint detection --> 

In [ ]:
diff, ratios_a, ratios_b = heatmap("og_photos/dana_a.jpg", "og_photos/dana_b.jpg")